# 59 — Tool Calling for Resume Tasks
**Goal:** Use function calling to route LLM decisions to specialized engines.

Tool calling (a.k.a. function calling) reverses the usual flow: the **LLM decides** which deterministic engine to invoke, and the engines — skill lookup, ATS scoring, bullet generation — stay as plain, testable Python. The model is the router; the code is the worker.

**Why it matters for resumes / ATS:** LLMs are unreliable at arithmetic and exact database lookups but excellent at understanding intent. Tool calling exploits that split: the model parses "does this resume match the JD?" into a `score_ats(resume_text, jd_text)` call with typed arguments, and the deterministic engine returns ground truth. Hallucination risk shrinks because the model never computes the score — it only selects the tool.

## 1. The Tool Pattern

The loop has four stages: the LLM analyzes the user's request, **decides** which tool to call, the tool executes a specialized engine, and the LLM formats the result for the user. The model is never asked to *do* the work the tool does — only to route to it.

**What the code does:** prints the pattern and catalogs the resume-domain tool set the rest of the block uses:
- `search_skills_db(query)` — canonical skill-name lookup (the taxonomy from Ch. 53)
- `parse_resume(file)` — structured extraction
- `score_ats(resume, jd)` — the rule engine from Ch. 50—52
- `get_skill_trends(skill)` — market data
- `generate_star_bullet(context)` — bullet generation (Ch. 60—61)

Each tool is a function with a *name* and a *description* — and that description is what the LLM reads when choosing, so writing it well is prompt engineering in disguise.

In [ ]:
print('''Tool calling pattern for resume AI:
1. LLM analyzes the user's request
2. LLM decides which tool to call (not the user)
3. Tool executes the specialized engine
4. LLM formats the result

Resume tools:
- search_skills_db(query) -> find canonical skill names
- parse_resume(file) -> extract structured data
- score_ats(resume, jd) -> compute match score
- get_skill_trends(skill_name) -> market data
- generate_star_bullet(context) -> bullet text''')

## 2. Tool Definitions

Tools are declared to the API as **JSON Schema** objects in the `tools` argument of the chat request. The schema is the contract the model must fill: it declares the function name, a natural-language description, the parameter types, and which parameters are required.

**What the code does:**
- `score_ats` — parameters `resume_text` (string) and `jd_text` (string), both `required`; the description tells the model when to reach for it: "Score a resume against a job description".
- `search_skills` — one parameter, `query`; "Search canonical skill taxonomy".
- The cell then prints each tool's name + description — **expected (verified by running):** two lines, `score_ats: Score a resume against a job description` and `search_skills: Search canonical skill taxonomy`.

**Why it matters:** the parameter schema is what the model fills in — a missing `required` list is how you get tool calls with no arguments. Declaring types (`string`) also lets the API reject malformed calls before your code sees them.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "score_ats",
            "description": "Score a resume against a job description",
            "parameters": {
                "type": "object",
                "properties": {
                    "resume_text": {"type": "string"},
                    "jd_text": {"type": "string"}
                },
                "required": ["resume_text", "jd_text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_skills",
            "description": "Search canonical skill taxonomy",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"]
            }
        }
    }
]
import json
print("Tool definitions ready for OpenAI API:")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

## 3. Tool Execution Engine

The schema is the interface; `execute_tool()` is the implementation. A real agent loop would: (1) send the request with `tools`, (2) get back `tool_calls` with a name plus arguments JSON, (3) call `execute_tool(name, args)`, (4) return the result to the model. This cell stubs the middle so the routing contract is testable without an API key.

**What the code does:** `execute_tool(name, args)` dispatches through a dict of lambdas — `score_ats` returns a hard-coded `{"score": 85, "confidence": 0.9}` (a stand-in for the Ch. 50—52 engine) and `search_skills` wraps its query as `"TensorFlow (canonical)"` (a stand-in for the Ch. 53 taxonomy). Unknown names hit the `{"error": ...}` fallback.

**Expected (verified by running):** `score_ats` returns `{'score': 85, 'confidence': 0.9}`, `search_skills` returns `{'results': ['TensorFlow (canonical)']}`, and an unknown tool returns `{'error': 'Unknown tool: nope'}` — the routing never raises, it reports.

In [ ]:
def execute_tool(name, args):
    """Route tool calls to actual engines."""
    tool_map = {
        "score_ats": lambda a: {"score": 85, "confidence": 0.9},
        "search_skills": lambda a: {"results": [a["query"] + " (canonical)"]},
    }
    fn = tool_map.get(name)
    if fn:
        return fn(args)
    return {"error": f"Unknown tool: {name}"}

print("Tool engine ready for routing:")
print(execute_tool("score_ats", {"resume_text": "...", "jd_text": "..."}))
print(execute_tool("search_skills", {"query": "TensorFlow"}))

## Summary: Tool calling lets LLMs delegate to specialized engines. Clean separation of concerns.

**The LLM decides, the code executes — routing intent to deterministic engines.**

Each capability is a JSON-Schema tool definition (name, description, typed parameters) backed by a plain Python function in `execute_tool()`. The model selects the tool and fills the arguments; the engine returns ground truth; unknown tools fail gracefully. Separation of concerns is the payoff: prompts can change without touching the scorers, and the scorers can be tested without the model.

Tool calling is the orchestration pattern Ch. 60 puts to work, where the LLM rewrites only the bullets the rule-based scorer flags as weak.